<a href="https://colab.research.google.com/github/putratogatorop/AI-Finance/blob/feat%2Frl-gym-environment/notebooks/rl_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RL Trading Agent — Population-Based Training

**Colab Pro runtime required.** Checkpoints save to Google Drive.

- Phase 1: 10 agents, 100 generations (~8 hours on T4)
- Phase 2: 20 agents, 500 generations (~40 hours)

**Setup:** Upload `rl_dataset.npz` to `My Drive/ai-finance/` before running.

In [6]:
# Patch agent.py — targeted find-and-replace fixes for NaN explosion
p = "/content/AI-Finance/services/python/src/rl/agent.py"
with open(p) as f: src = f.read()

# Fix 1: Add LayerNorm after first Linear in actor_mean
src = src.replace(
    "nn.Linear(hidden_size, 128),\n            nn.ReLU(),",
    "nn.Linear(hidden_size, 128),\n            nn.LayerNorm(128),\n            nn.ReLU(),"
)

# Fix 2: Add small weight init after actor_mean definition
src = src.replace(
    "# Learnable log standard deviation",
    "# Initialize output layer with small weights to prevent Tanh saturation\n        nn.init.uniform_(self.actor_mean[-2].weight, -0.003, 0.003)\n        nn.init.zeros_(self.actor_mean[-2].bias)\n\n        # Learnable log standard deviation (conservative init)"
)

# Fix 3: Change log_std init from zeros to -0.5
src = src.replace("torch.zeros(self.action_size)", "torch.full((self.action_size,), -0.5)")

# Fix 4: Add clamp to log_std in act() and evaluate()
src = src.replace(
    "action_std = torch.exp(self.actor_log_std)",
    "action_std = torch.exp(\n            self.actor_log_std.clamp(min=-5.0, max=2.0)\n        ).clamp(min=0.01)\n        action_mean = torch.nan_to_num(action_mean, nan=0.0)\n        # replaced orig action_std"
)

# Fix 5: Same for evaluate method — handle the second occurrence
src = src.replace(
    "action_std = torch.exp(self.actor_log_std).expand_as(action_mean)",
    "action_std = torch.exp(\n            self.actor_log_std.clamp(min=-5.0, max=2.0)\n        ).clamp(min=0.01).expand_as(action_mean)\n        action_mean = torch.nan_to_num(action_mean, nan=0.0)"
)

with open(p, 'w') as f: f.write(src)

# Verify
with open(p) as f: t = f.read()
checks = {
    "LayerNorm": "LayerNorm" in t,
    "small_init": "-0.003, 0.003" in t,
    "log_std_-0.5": "full((self.action_size,), -0.5)" in t,
    "clamp": "clamp(min=-5.0" in t,
    "nan_guard": "nan_to_num" in t,
}
for k, v in checks.items():
    print(f"  {k}: {'OK' if v else 'MISSING'}")
print("All fixes applied!" if all(checks.values()) else "WARNING: Some fixes missing — check agent.py")

FileNotFoundError: [Errno 2] No such file or directory: '/content/AI-Finance/services/python/src/rl/agent.py'

In [10]:
from google.colab import drive
drive.mount('/content/drive')

# Clone repo (or pull if already exists)
import os
if not os.path.exists("/content/AI-Finance"):
    !git clone https://github.com/putratogatorop/AI-Finance.git /content/AI-Finance
else:
    %cd /content/AI-Finance
    !git pull

%cd /content/AI-Finance
!git checkout feat/rl-gym-environment
%cd /content/AI-Finance/services/python
!pip install -q torch numpy pandas

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/AI-Finance
fatal: not a git repository (or any of the parent directories): .git
/content/AI-Finance
fatal: not a git repository (or any of the parent directories): .git
[Errno 2] No such file or directory: '/content/AI-Finance/services/python'
/content/AI-Finance


In [11]:
!rm -rf /content/AI-Finance

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import os, subprocess

# Clone public repo (no auth needed)
if not os.path.exists("/content/AI-Finance"):
    subprocess.run(["git", "clone", "https://github.com/putratogatorop/AI-Finance.git", "/content/AI-Finance"], check=True)

    # Checkout branch, install deps
    os.chdir("/content/AI-Finance")
    subprocess.run(["git", "checkout", "feat/rl-gym-environment"], check=True)
    os.chdir("/content/AI-Finance/services/python")
    subprocess.run(["pip", "install", "-q", "torch", "numpy", "pandas"], check=True)
    print("Ready! Repo cloned, branch checked out, deps installed.")

CalledProcessError: Command '['git', 'clone', 'https://github.com/putratogatorop/AI-Finance.git', '/content/AI-Finance']' returned non-zero exit status 128.

In [14]:
!git clone https://github.com/putratogatorop/AI-Finance.git /content/AI-Finance 2>&1

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/AI-Finance'...
fatal: Unable to read current working directory: No such file or directory


In [18]:
import os, subprocess, shutil

# Fix broken CWD from previous cell
os.chdir("/content")

# Clean any leftover directory
if os.path.exists("/content/AI-Finance"):
    shutil.rmtree("/content/AI-Finance")

    # Clone fresh
    subprocess.run(["git", "clone", "https://github.com/putratogatorop/AI-Finance.git", "/content/AI-Finance"], check=True)

    # Checkout branch and install deps
    os.chdir("/content/AI-Finance")
    subprocess.run(["git", "checkout", "feat/rl-gym-environment"], check=True)
    os.chdir("/content/AI-Finance/services/python")
    subprocess.run(["pip", "install", "-q", "torch", "numpy", "pandas"], check=True)
    print("Ready! Repo cloned, branch checked out, deps installed.")

Ready! Repo cloned, branch checked out, deps installed.


In [16]:
import numpy as np
from pathlib import Path

# Copy dataset from Google Drive (upload rl_dataset.npz to My Drive/ai-finance/ first)
DRIVE_DATA = Path("/content/drive/MyDrive/AI-Finance/rl_dataset.npz")
LOCAL_DATA = Path("/content/AI-Finance/data/features/rl_dataset.npz")
LOCAL_DATA.parent.mkdir(parents=True, exist_ok=True)

assert DRIVE_DATA.exists(), f"Dataset not found at {DRIVE_DATA}. Upload rl_dataset.npz to My Drive/ai-finance/"
!cp "{DRIVE_DATA}" "{LOCAL_DATA}"

d = np.load(str(LOCAL_DATA), allow_pickle=True)
print(f"Dataset loaded: {d['alt_features'].shape[1]} alts, {d['alt_features'].shape[0]} timestamps, {d['alt_features'].shape[2]} features")

Dataset loaded: 178 alts, 3033 timestamps, 48 features


In [ ]:
!ls /content/drive/MyDrive/AI-Finance/ 2>/dev/null || echo "Folder not found"; find /content/drive/MyDrive -name "rl_dataset*" -type f 2>/dev/null | head -5

In [ ]:
CKPT = "/content/drive/MyDrive/AI-Finance/rl_checkpoints"
!python scripts/train_rl.py --data "{LOCAL_DATA}" --checkpoint {CKPT} --population 10 --generations 100 --episodes 10

In [ ]:
# Patch agent.py with NaN fix — write fixed file directly
import textwrap, pathlib
p = pathlib.Path("/content/AI-Finance/services/python/src/rl/agent.py")
src = p.read_text()
# Verify we have the unfixed version
assert "LayerNorm" not in src or True, "Already patched"
p.write_text(textwrap.dedent('''\
"""LSTM + PPO actor-critic agent network for RL crypto trading."""

from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal


class LSTMPPOAgent(nn.Module):
    """Actor-critic network with LSTM backbone for PPO training."""

    def __init__(
        self,
        obs_size: int,
        n_alts: int,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.3,
    ) -> None:
        super().__init__()
        self.obs_size = obs_size
        self.n_alts = n_alts
        self.action_size = n_alts * 3
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=obs_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.actor_mean = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Linear(128, self.action_size),
            nn.Tanh(),
        )

        nn.init.uniform_(self.actor_mean[-2].weight, -0.003, 0.003)
        nn.init.zeros_(self.actor_mean[-2].bias)

        self.actor_log_std = nn.Parameter(
            torch.full((self.action_size,), -0.5)
        )

        self.critic = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )

        self.hidden: tuple[torch.Tensor, torch.Tensor] | None = None

    def reset_hidden(self) -> None:
        self.hidden = None

    def _forward_lstm(self, obs: torch.Tensor) -> torch.Tensor:
        if obs.dim() == 1:
            obs = obs.unsqueeze(0).unsqueeze(0)
        elif obs.dim() == 2:
            obs = obs.unsqueeze(1)
        lstm_out, self.hidden = self.lstm(obs, self.hidden)
        return lstm_out[:, -1, :]

    @torch.no_grad()
    def act(self, obs: np.ndarray, deterministic: bool = False) -> tuple[np.ndarray, float, float]:
        obs_t = torch.as_tensor(obs, dtype=torch.float32)
        h = self._forward_lstm(obs_t)
        action_mean = self.actor_mean(h).squeeze(0)
        action_std = torch.exp(self.actor_log_std.clamp(min=-5.0, max=2.0)).clamp(min=0.01)
        action_mean = torch.nan_to_num(action_mean, nan=0.0)
        dist = Normal(action_mean, action_std)
        if deterministic:
            action = action_mean
        else:
            action = dist.sample()
        log_prob = dist.log_prob(action).sum().item()
        value = self.critic(h).squeeze().item()
        action_np = action.cpu().numpy()
        for i in range(self.n_alts):
            base = i * 3
            action_np[base] = np.clip(action_np[base], -1.0, 1.0)
            action_np[base + 1] = np.clip(action_np[base + 1], 0.0, 1.0)
            action_np[base + 2] = np.clip(action_np[base + 2], 0.0, 1.0)
        return action_np, log_prob, value

    def evaluate(self, obs_batch: torch.Tensor, action_batch: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        saved_hidden = self.hidden
        self.hidden = None
        h = self._forward_lstm(obs_batch)
        action_mean = self.actor_mean(h)
        action_std = torch.exp(self.actor_log_std.clamp(min=-5.0, max=2.0)).clamp(min=0.01)
        action_mean = torch.nan_to_num(action_mean, nan=0.0)
        dist = Normal(action_mean, action_std)
        log_probs = dist.log_prob(action_batch).sum(dim=-1)
        values = self.critic(h).squeeze(-1)
        entropy = dist.entropy().mean()
        self.hidden = saved_hidden
        return log_probs, values, entropy
'''))
print("Patched! Verifying...")
t = p.read_text()
print(f"LayerNorm: {'LayerNorm' in t}")
print(f"nan_to_num: {'nan_to_num' in t}")
print(f"clamp -5.0: {'clamp(min=-5.0' in t}")
print(f"full -0.5: {'full((self.action_size,), -0.5)' in t}")
print("All fixes applied!" if all(x in t for x in ["LayerNorm","nan_to_num","clamp(min=-5.0","full((self.action_size,), -0.5)"]) else "MISSING FIXES")

In [ ]:
import json
from pathlib import Path
meta = Path(CKPT) / "metadata.json"
if meta.exists():
    with open(meta) as f: m = json.load(f)
    print(f"Gen: {m['generation']}, Best: {m['best_reward']:.4f}, Mean: {m['mean_reward']:.4f}")

In [ ]:
# !python scripts/train_rl.py --data "{LOCAL_DATA}" --checkpoint {CKPT} --population 20 --generations 500 --episodes 10

In [ ]:
# !python scripts/evaluate_rl.py --data "{LOCAL_DATA}" --checkpoint {CKPT}

In [19]:
# Fix 2: Add nan_to_num guard on action_std (clamp doesn't fix NaN)
p = "/content/AI-Finance/services/python/src/rl/agent.py"
with open(p) as f: src = f.read()

# Replace action_std line to add nan_to_num protection
old = "action_std = torch.exp(self.actor_log_std.clamp(min=-5.0, max=2.0)).clamp(min=0.01)"
new = "action_std = torch.nan_to_num(torch.exp(self.actor_log_std.clamp(min=-5.0, max=2.0)), nan=0.01).clamp(min=0.01)"
src = src.replace(old, new)

with open(p, 'w') as f: f.write(src)

# Also need to fix the gradient issue: add gradient clipping to actor_log_std
# The root cause is likely NaN gradients corrupting the parameter
# Let's also check ppo.py for gradient clipping
with open(p) as f: t = f.read()
count = t.count("nan_to_num")
print(f"nan_to_num occurrences: {count}")
print(f"action_std fix applied: {'torch.nan_to_num(torch.exp' in t}")
print("Done!")

nan_to_num occurrences: 0
action_std fix applied: False
Done!


In [20]:
import re

p = "/content/AI-Finance/services/python/src/rl/agent.py"
with open(p) as f:
    src = f.read()

    # Show what we're working with
    print("Before fix:")
    for i, line in enumerate(src.split('\n')):
        if 'action_std' in line:
                print(f"  Line {i}: {line.strip()}")

                # Use regex to replace multi-line action_std pattern
                # Match: action_std = torch.exp(\n            self.actor_log_std.clamp(...)\n        ).clamp(min=0.01)
                pattern = r'action_std = torch\.exp\(\s*\n\s*self\.actor_log_std\.clamp\(min=-5\.0, max=2\.0\)\s*\n\s*\)\.clamp\(min=0\.01\)'
                replacement = 'action_std = torch.nan_to_num(\n            torch.exp(self.actor_log_std.clamp(min=-5.0, max=2.0)),\n            nan=0.01,\n        ).clamp(min=0.01)'

                new_src = re.sub(pattern, replacement, src)

                # Also try single-line pattern just in case
                old_single = "action_std = torch.exp(self.actor_log_std.clamp(min=-5.0, max=2.0)).clamp(min=0.01)"
                new_single = "action_std = torch.nan_to_num(torch.exp(self.actor_log_std.clamp(min=-5.0, max=2.0)), nan=0.01).clamp(min=0.01)"
                new_src = new_src.replace(old_single, new_single)

                with open(p, 'w') as f:
                    f.write(new_src)

                    # Verify
                    with open(p) as f:
                        t = f.read()
                        count = t.count("nan_to_num")
                        print(f"\nAfter fix:")
                        print(f"nan_to_num occurrences: {count}")
                        for i, line in enumerate(t.split('\n')):
                            if 'action_std' in line:
                                    print(f"  Line {i}: {line.strip()}")
                                    print("Done!" if count >= 2 else "WARNING: fix may not have applied correctly")

Before fix:
  Line 100: action_std = torch.exp(self.actor_log_std).clamp(min=0.01)

After fix:
nan_to_num occurrences: 0
  Line 101: dist = Normal(action_mean, action_std)

After fix:
nan_to_num occurrences: 0
  Line 141: action_std = torch.exp(self.actor_log_std).clamp(min=0.01)

After fix:
nan_to_num occurrences: 0
  Line 142: dist = Normal(action_mean, action_std)

After fix:
nan_to_num occurrences: 0


In [21]:
# Fix 3: Correct pattern - the GitHub code uses simpler format
p = "/content/AI-Finance/services/python/src/rl/agent.py"
with open(p) as f:
    src = f.read()

    old = "action_std = torch.exp(self.actor_log_std).clamp(min=0.01)"
    new = "action_std = torch.nan_to_num(\n            torch.exp(self.actor_log_std.clamp(min=-5.0, max=2.0)),\n            nan=0.01,\n        ).clamp(min=0.01)"

    count_before = src.count(old)
    print(f"Found {count_before} occurrences of old pattern")

    src = src.replace(old, new)

    # Also add nan_to_num guard on action_mean if not present
    if "action_mean = torch.nan_to_num" not in src:
        src = src.replace(
                "dist = Normal(action_mean, action_std)",
                        "action_mean = torch.nan_to_num(action_mean, nan=0.0)\n        dist = Normal(action_mean, action_std)"
                            )

                            with open(p, 'w') as f:
                                f.write(src)

                                # Verify
                                with open(p) as f:
                                    t = f.read()
                                    count = t.count("nan_to_num")
                                    print(f"nan_to_num occurrences after fix: {count}")
                                    for i, line in enumerate(t.split('\n')):
                                        if 'nan_to_num' in line or 'action_std' in line:
                                                print(f"  Line {i}: {line.rstrip()}")
                                                print("\nDone!" if count >= 4 else "\nWARNING: expected at least 4 nan_to_num occurrences")

IndentationError: unexpected indent (2020472606.py, line 21)

In [22]:
p="/content/AI-Finance/services/python/src/rl/agent.py"; f=open(p); s=f.read(); f.close(); old="action_std = torch.exp(self.actor_log_std).clamp(min=0.01)"; new="action_std = torch.nan_to_num(torch.exp(self.actor_log_std.clamp(min=-5.0, max=2.0)), nan=0.01).clamp(min=0.01)"; s=s.replace(old,new); s=s.replace("dist = Normal(action_mean, action_std)","action_mean = torch.nan_to_num(action_mean, nan=0.0)\n        dist = Normal(action_mean, action_std)") if "torch.nan_to_num(action_mean" not in s else s; f=open(p,'w'); f.write(s); f.close(); f=open(p); t=f.read(); f.close(); print(f"nan_to_num count: {t.count('nan_to_num')}"); [print(f"  L{i}: {l.strip()}") for i,l in enumerate(t.split(chr(10))) if 'nan_to_num' in l])"

SyntaxError: unmatched ')' (30695830.py, line 1)

In [23]:
%%writefile /content/fix_agent.py
import sys
p = "/content/AI-Finance/services/python/src/rl/agent.py"
with open(p) as f:
    s = f.read()
    old = "action_std = torch.exp(self.actor_log_std).clamp(min=0.01)"
    new = "action_std = torch.nan_to_num(torch.exp(self.actor_log_std.clamp(min=-5.0, max=2.0)), nan=0.01).clamp(min=0.01)"
    n = s.count(old)
    s = s.replace(old, new)
    if "torch.nan_to_num(action_mean" not in s:
        s = s.replace("dist = Normal(action_mean, action_std)", "action_mean = torch.nan_to_num(action_mean, nan=0.0)\n        dist = Normal(action_mean, action_std)")
        with open(p, "w") as f:
            f.write(s)
            with open(p) as f:
                t = f.read()
                c = t.count("nan_to_num")
                print(f"Replaced {n} action_std lines. nan_to_num total: {c}")
                for i, line in enumerate(t.split("\n")):
                    if "nan_to_num" in line:
                            print(f"  L{i}: {line.strip()}")

Writing /content/fix_agent.py


In [ ]:

path = "/content/AI-Finance/services/python/src/rl/agent.py"
with open(path, "r") as f:
    code = f.read()

    # Add LayerNorm after first nn.Linear in actor_mean Sequential
    old = """self.actor_mean = nn.Sequential(
                nn.Linear(hidden_size, 128),
                            nn.ReLU(),"""

                            new = """self.actor_mean = nn.Sequential(
                                        nn.Linear(hidden_size, 128),
                                                    nn.LayerNorm(128),
                                                                nn.ReLU(),"""

                                                                if old in code:
                                                                    code = code.replace(old, new)
                                                                        with open(path, "w") as f:
                                                                                f.write(code)
                                                                                    print("SUCCESS: LayerNorm added to actor_mean Sequential")
                                                                                    else:
                                                                                        print("ERROR: Pattern not found. Current actor_mean section:")
                                                                                            idx = code.find("actor_mean")
                                                                                                if idx >= 0:
                                                                                                        print(code[idx:idx+300])
                                                                                                        PYEOF
